In [83]:
from PIL import Image
import imagehash
import os
import pybktree
import collections
import requests
from webdav3.client import Client
from requests.auth import HTTPBasicAuth
from io import BytesIO
from IPython import display
from IPython.display import display

In [84]:
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, MetaData, Table, select, insert
from sqlalchemy.exc import SQLAlchemyError
load_dotenv()
import pandas as pd



def create_db_connection():
    load_dotenv()
    DB_HOST = os.getenv("DB_HOST")
    DB_NAME = os.getenv("DB_NAME")
    DB_USER = os.getenv("DB_USER")
    DB_PASSWORD = os.getenv("DB_PASSWORD")
    engine = create_engine('postgresql+pg8000://'+DB_USER+':'+DB_PASSWORD+'@'+DB_HOST+':5432/'+DB_NAME)
    return engine


In [85]:
def get_preview_index(preview_size, DB_HOST):
    """
    Get preview index from bre_advance_index, filtering out already-hashed files.
    
    Parameters:
    -----------
    preview_size : int
        Size for preview URL generation
    DB_HOST : str
        Database host for URL construction
    
    Returns:
    --------
    pandas.DataFrame
        Filtered DataFrame with only new file_ids not in bre_hashes table
    """
    engine = create_db_connection()
    metadata = MetaData()

    # Reflect the tables
    Index_table = Table('bre_advance_index', metadata, autoload_with=engine)
    Hash_table = Table('bre_hashes', metadata, autoload_with=engine)
    
    # Get all existing file_ids from the bre_hashes table
    with engine.connect() as connection:
        existing_query = select(Hash_table.c.id)
        existing_ids = set(connection.execute(existing_query).fetchall())
        existing_ids = {row[0] for row in existing_ids}
    
    # Query the advance index table
    query = select(Index_table)
    with engine.connect() as connection:
        df_index = pd.DataFrame(connection.execute(query).fetchall())
        # Set column names
        df_index.columns = Index_table.columns.keys()
    
    # Filter to only include file_ids that don't exist in bre_hashes
    # Assuming 'fileid' column contains the file IDs
    df_index['fileid'] = df_index['fileid'].astype(str)
    df_index = df_index[~df_index['fileid'].isin(existing_ids)]
    
    # Update preview URLs
    for index, row in df_index.iterrows():
        prev_url = row['preview_url']
        prev_url = f'''http://{DB_HOST}:8080{prev_url.replace('{prevsize}', f'''x={preview_size}&y={preview_size}''')}'''
        df_index.loc[index, 'preview_url'] = prev_url
    
    print(f"Filtered index: {len(df_index)} new files out of total")
    return df_index


In [86]:
df_index = get_preview_index(540, os.getenv("DB_HOST"))


Filtered index: 1507 new files out of total


In [87]:
df_index = df_index.head(10)

In [88]:
# In-memory cache for downloaded files (file_id -> PIL Image)
image_cache = {}

def get_images(file_id, file_path, webdav_path=None):
    DB_HOST = os.getenv("DB_HOST")
    NC_ACC = os.getenv("NC_ACC")
    NC_PASS = os.getenv("NC_PASS")

    # Check cache first
    if file_id in image_cache:
        return file_id, image_cache[file_id]

    # Send a GET request to download the preview
    response = requests.get(file_path, auth=HTTPBasicAuth(NC_ACC, NC_PASS), stream=True)
  
    try:
        if response.status_code == 200:
            file_in_memory = BytesIO()
            for chunk in response.iter_content(chunk_size=1024):
                if chunk:
                    file_in_memory.write(chunk)
            file_in_memory.seek(0) 
            img = Image.open(file_in_memory)
            img = img.resize((540, 540))
            image_cache[file_id] = img  # Cache the image
            return file_id, img
        elif response.status_code == 404:
            print(f"No preview available for file: {file_id}, loading original via WebDAV...")
            # Fall back to WebDAV if webdav_path is provided
            if webdav_path is not None:
                webdav_base_url = f'http://{DB_HOST}:8080/remote.php/dav/files/{NC_ACC}'
                webdav_file_url = f'{webdav_base_url}{webdav_path}'
                
                try:
                    webdav_response = requests.get(webdav_file_url, auth=HTTPBasicAuth(NC_ACC, NC_PASS), stream=True)
                    if webdav_response.status_code == 200:
                        file_in_memory = BytesIO()
                        for chunk in webdav_response.iter_content(chunk_size=1024):
                            if chunk:
                                file_in_memory.write(chunk)
                        file_in_memory.seek(0)
                        source_img = Image.open(file_in_memory)
                        img = source_img.resize((540, 540))
                        image_cache[file_id] = img  # Cache the image
                        print(f"Successfully loaded file {file_id} via WebDAV")
                        return file_id, img
                    else:
                        print(f"Failed to download via WebDAV. Status code: {webdav_response.status_code}")
                except Exception as e:
                    print(f"Error downloading via WebDAV for file {file_id}: {e}")
            return file_id, None
        else:
            print(f"Failed to download file. Status code: {response.status_code}")
            print(response.text)
            return file_id, None
    except Exception as e:
        print(f"Error downloading file {file_id}: {e}")
        return file_id, None

In [89]:
for row in df_index.iloc():
    file_id = row['fileid']
    file_path = row['preview_url']
    webdav_path = row.get('path')  # WebDAV path to original file
    file_id, img = get_images(file_id, file_path, webdav_path)
    if img is None:
        print(f"Failed to download file: {file_id}")
    else:
        whash = imagehash.whash(img)
        ahash = imagehash.average_hash(img)
        phash = imagehash.phash(img)
        df_index.loc[row.name, "w_hash"] = whash
        df_index.loc[row.name, "a_hash"] = ahash
        df_index.loc[row.name, "p_hash"] = phash


No preview available for file: 2555, loading original via WebDAV...
Successfully loaded file 2555 via WebDAV


In [90]:
df_index= df_index.drop(columns=['path', 'preview_url', 'id','name'])

In [91]:
df_index

,fileid,w_hash,a_hash,p_hash
10,2140,fffbfb3c18c00104,fbfbfb3810800000,cdcd9d181692f495
11,2555,fffffebc90010080,ffbef63c10000080,ded4912bc11be16c
12,2155,ffe3c3818080e1f7,fff7e3c18191f7ff,eea3c31839e2cd1c
13,2156,3dc3e41919c3fb81,bdc1e43999c3fb91,ecd2ab2d0b8374d2
14,2139,632177ad216321f7,ef2dd6ad01ff29d6,ab8b948bd4abd454
15,2146,0ded04ad4dedb24c,48edb7004ded3248,a2ea28fa382d8dae
16,2144,792073d84d0963bf,792073d80f9fe39f,a843ef1bd9e485a4
17,2145,ff9100003dc2f4ff,ff9b01003dc3feff,ec2df1f283542c4b
18,2143,c2ff9301b9467c19,c2ffd38199c67c99,fc34d4d02d703c8f
19,2151,2cbd18db04d308ff,3cbd18db64c399bd,9819cfb393497913


In [92]:
def push_index(df_index):
    """
    Push and append data from df_index to the bre_hashes table in PostgreSQL.
    
    Parameters:
    -----------
    df_index : pandas.DataFrame
        DataFrame with columns: 'fileid', 'w_hash', 'a_hash', 'p_hash'
    """
    DB_HOST = os.getenv("DB_HOST")
    NC_ACC = os.getenv("NC_ACC")
    NC_PASS = os.getenv("NC_PASS")
    engine = create_db_connection()
    metadata = MetaData()

    # Reflect the table
    Index_table = Table('bre_hashes', metadata, autoload_with=engine)
    
    # Rename DataFrame columns to match database columns
    df_mapped = df_index.rename(columns={'fileid': 'id'})
    
    # Convert DataFrame to list of dictionaries for bulk insert
    data_to_insert = df_mapped.to_dict('records')
    
    # Insert data into the database
    with engine.connect() as connection:
        if data_to_insert:  # Only insert if there's data
            connection.execute(insert(Index_table), data_to_insert)
            connection.commit()
    
    print(f"Successfully appended {len(data_to_insert)} rows to bre_hashes table")
    return len(data_to_insert)


In [93]:
push_index(df_index)

Successfully appended 10 rows to bre_hashes table


10